# QaptaanLM-0.75B-Instruct: Stage 2 SFT Hub Publisher
### Package, Validate & Publish SFT Model to Hugging Face (`kaptaan45/QaptaanLM-0.75B-Instruct`)

This notebook loads the fine-tuned Stage 2 SFT model checkpoint from the Kaggle dataset (`kaptaan45/checkpoints-sft`), verifies all parameter tensors, builds the official standalone Hugging Face repository package with custom modeling and complete Model Card, tests inference, and uploads to the Hugging Face Hub.

| Asset | Target / Source |
|:---|:---|
| **Source SFT Dataset** | [kaptaan45/checkpoints-sft](https://www.kaggle.com/datasets/kaptaan45/checkpoints-sft) |
| **Target HF Repository** | [kaptaan45/QaptaanLM-0.75B-Instruct](https://huggingface.co/kaptaan45/QaptaanLM-0.75B-Instruct) |
| **Parameters** | 752M dense parameters (text-only hybrid DeltaNet + GQA) |
| **Format** | PyTorch `model.safetensors` with custom `modeling_qaptaan.py` and `configuration_qaptaan.py` |

## 1. Install & Setup Dependencies

In [ ]:
!pip install -q --upgrade huggingface_hub safetensors transformers accelerate kaggle


## 2. Authenticate Hugging Face & Kaggle

In [ ]:
import os, json, getpass
from huggingface_hub import HfApi, login

# Hugging Face Token setup (retrieves from Kaggle Secrets, env var, or interactive prompt)
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    HF_TOKEN = input("Enter your Hugging Face write token (hf_...): ").strip()

login(token=HF_TOKEN, add_to_git_credential=True)
print("✓ Logged in to Hugging Face successfully!")

# Kaggle Credentials setup (for downloading dataset if run outside Kaggle or without attached dataset)
KAGGLE_USER = os.environ.get("KAGGLE_USERNAME", "kaptaan45")
KAGGLE_KEY = os.environ.get("KAGGLE_KEY", "")
try:
    from kaggle_secrets import UserSecretsClient
    KAGGLE_KEY = UserSecretsClient().get_secret("KAGGLE_KEY") or KAGGLE_KEY
except Exception:
    pass
if KAGGLE_KEY:
    os.environ["KAGGLE_USERNAME"] = KAGGLE_USER
    os.environ["KAGGLE_KEY"] = KAGGLE_KEY
print(f"✓ Kaggle credentials configured for user: {KAGGLE_USER}")


## 3. Locate or Download the SFT Checkpoint Dataset

In [ ]:
import glob, shutil
from pathlib import Path

# Candidate input paths where Kaggle mounts datasets or local directories
candidate_paths = [
    "/kaggle/input/checkpoints-sft/jax_sft_hf",
    "/kaggle/input/checkpoints-sft",
    "/kaggle/input/checkpoints_sft/jax_sft_hf",
    "/kaggle/input/checkpoints_sft",
    "/kaggle/working/checkpoints/jax_sft_hf",
    "checkpoints/jax_sft_hf",
] + glob.glob("/kaggle/input/*sft*/**", recursive=True)

source_dir = None
for cp in candidate_paths:
    p = Path(cp)
    if p.exists() and (p.is_dir() or p.name.endswith(".safetensors")):
        if (p / "model.safetensors").exists() or (p / "config.json").exists() or p.name.endswith(".safetensors") or (p / "state").exists():
            source_dir = p if p.is_dir() else p.parent
            break

# If not mounted, download dataset using Kaggle API
if source_dir is None:
    print("SFT checkpoint not found in local paths. Downloading via Kaggle API...")
    import kaggle
    download_target = Path("/kaggle/working/checkpoints_sft_download") if Path("/kaggle/working").exists() else Path("./checkpoints_sft_download")
    download_target.mkdir(parents=True, exist_ok=True)
    kaggle.api.dataset_download_files("kaptaan45/checkpoints-sft", path=str(download_target), unzip=True)
    source_dir = download_target
    print(f"✓ Downloaded dataset to: {source_dir}")
else:
    print(f"✓ Located SFT checkpoint dataset at: {source_dir}")

print("Files in source checkpoint:")
for f in source_dir.glob("**/*"):
    if f.is_file():
        print(f" - {f.relative_to(source_dir)} ({f.stat().st_size / (1024*1024):.2f} MB)")


## 4. Build Complete, Standalone Hugging Face Model Package

In [ ]:
import os, sys, json, shutil
from pathlib import Path
from safetensors.torch import save_file, load_file
import torch
from transformers import AutoTokenizer

export_dir = Path("/kaggle/working/qaptaanlm_instruct_hf") if Path("/kaggle/working").exists() else Path("./qaptaanlm_instruct_hf")
export_dir.mkdir(parents=True, exist_ok=True)

print(f"Packaging model repository into: {export_dir.resolve()}")

# 1. Handle model weights (safetensors)
st_candidates = list(source_dir.glob("**/*.safetensors"))
if st_candidates:
    st_src = st_candidates[0]
    print(f"Copying safetensors weights from {st_src}...")
    shutil.copy2(st_src, export_dir / "model.safetensors")
    print(f"✓ Saved model.safetensors ({ (export_dir / 'model.safetensors').stat().st_size / (1024*1024):.2f} MB)")
else:
    # If Orbax / JAX checkpoint state exists, convert Flax params to safetensors
    print("Converting JAX checkpoint state to safetensors...")
    import orbax.checkpoint as ocp
    from jax_training.models.config import Qwen3_5Config
    from jax_training.models.convert import convert_flax_to_pytorch_state_dict
    from safetensors.numpy import save_file as save_numpy_file
    
    state_path = source_dir / "state" if (source_dir / "state").exists() else source_dir
    checkpointer = ocp.StandardCheckpointer()
    restored = checkpointer.restore(state_path)
    params = restored["params"]
    config_c = Qwen3_5Config(dtype="bfloat16")
    state_dict = convert_flax_to_pytorch_state_dict(params, config_c)
    save_numpy_file(state_dict, str(export_dir / "model.safetensors"))
    print("✓ Converted and saved model.safetensors!")

# 2. Write configuration_qaptaan.py
config_py_code = '''"""QaptaanLM-0.75B-Instruct Configuration."""

from transformers.configuration_utils import PretrainedConfig


class QaptaanConfig(PretrainedConfig):
    model_type = "qaptaan"
    keys_to_ignore_at_inference = ["past_key_values"]

    def __init__(
        self,
        vocab_size: int = 248320,
        hidden_size: int = 1024,
        intermediate_size: int = 3584,
        num_hidden_layers: int = 24,
        num_attention_heads: int = 8,
        num_key_value_heads: int = 2,
        head_dim: int = 256,
        rms_norm_eps: float = 1e-6,
        tie_word_embeddings: bool = True,
        max_position_embeddings: int = 262144,
        rope_theta: float = 10000000.0,
        partial_rotary_factor: float = 0.25,
        attn_output_gate: bool = True,
        full_attention_interval: int = 4,
        linear_key_head_dim: int = 128,
        linear_value_head_dim: int = 128,
        linear_num_key_heads: int = 16,
        linear_num_value_heads: int = 16,
        linear_conv_kernel_dim: int = 4,
        hidden_act: str = "silu",
        initializer_range: float = 0.02,
        use_cache: bool = True,
        bos_token_id: int = None,
        eos_token_id: int = 151645,
        pad_token_id: int = 151643,
        **kwargs,
    ):
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.intermediate_size = intermediate_size
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.num_key_value_heads = num_key_value_heads
        self.head_dim = head_dim
        self.rms_norm_eps = rms_norm_eps
        self.tie_word_embeddings = tie_word_embeddings
        self.max_position_embeddings = max_position_embeddings
        self.rope_theta = rope_theta
        self.partial_rotary_factor = partial_rotary_factor
        self.attn_output_gate = attn_output_gate
        self.full_attention_interval = full_attention_interval
        self.linear_key_head_dim = linear_key_head_dim
        self.linear_value_head_dim = linear_value_head_dim
        self.linear_num_key_heads = linear_num_key_heads
        self.linear_num_value_heads = linear_num_value_heads
        self.linear_conv_kernel_dim = linear_conv_kernel_dim
        self.hidden_act = hidden_act
        self.initializer_range = initializer_range
        self.use_cache = use_cache

        self.layer_types = []
        for i in range(num_hidden_layers):
            if (i + 1) % full_attention_interval == 0:
                self.layer_types.append("full_attention")
            else:
                self.layer_types.append("linear_attention")

        super().__init__(
            bos_token_id=bos_token_id,
            eos_token_id=eos_token_id,
            pad_token_id=pad_token_id,
            tie_word_embeddings=tie_word_embeddings,
            **kwargs,
        )
'''
with open(export_dir / "configuration_qaptaan.py", "w", encoding="utf-8") as f:
    f.write(config_py_code)
print("✓ Created configuration_qaptaan.py")

# 3. Write modeling_qaptaan.py
from scripts.rebuild_and_upload_hf import build_modeling_qaptaan_code
with open(export_dir / "modeling_qaptaan.py", "w", encoding="utf-8") as f:
    f.write(build_modeling_qaptaan_code())
print("✓ Created modeling_qaptaan.py")

# 4. Write config.json
config_dict = {
    "architectures": ["QaptaanForCausalLM"],
    "model_type": "qaptaan",
    "auto_map": {
        "AutoConfig": "configuration_qaptaan.QaptaanConfig",
        "AutoModelForCausalLM": "modeling_qaptaan.QaptaanForCausalLM"
    },
    "vocab_size": 248320,
    "hidden_size": 1024,
    "intermediate_size": 3584,
    "num_hidden_layers": 24,
    "num_attention_heads": 8,
    "num_key_value_heads": 2,
    "head_dim": 256,
    "rms_norm_eps": 1e-6,
    "tie_word_embeddings": True,
    "max_position_embeddings": 262144,
    "rope_theta": 10000000.0,
    "partial_rotary_factor": 0.25,
    "attn_output_gate": True,
    "full_attention_interval": 4,
    "linear_key_head_dim": 128,
    "linear_value_head_dim": 128,
    "linear_num_key_heads": 16,
    "linear_num_value_heads": 16,
    "linear_conv_kernel_dim": 4,
    "layer_types": [
        "full_attention" if (i + 1) % 4 == 0 else "linear_attention" for i in range(24)
    ],
    "use_cache": True,
    "torch_dtype": "bfloat16"
}
with open(export_dir / "config.json", "w", encoding="utf-8") as f:
    json.dump(config_dict, f, indent=2)
print("✓ Created config.json")

# 5. Write generation_config.json
gen_config = {
    "bos_token_id": None,
    "eos_token_id": [151645, 151643],
    "pad_token_id": 151643,
    "do_sample": False,
    "temperature": 0.2,
    "top_p": 0.95,
    "repetition_penalty": 1.05,
    "transformers_version": "4.49.0"
}
with open(export_dir / "generation_config.json", "w", encoding="utf-8") as f:
    json.dump(gen_config, f, indent=2)
print("✓ Created generation_config.json")

# 6. Download and save official Qwen3.5 tokenizer
print("Downloading official Qwen3.5 Tokenizer files...")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-0.8B-Base", trust_remote_code=True)
tokenizer.save_pretrained(str(export_dir))
print("✓ Saved official tokenizer files (tokenizer.json, vocab.json, merges.txt, chat_template)")

# 7. Write complete, professional README.md (Model Card)
instruct_model_card = '''---
license: apache-2.0
base_model: kaptaan45/QaptaanLM-0.75B
language:
- en
- code
tags:
- code
- causal-lm
- qwen3.5
- hybrid-attention
- deltanet
- gqa
- instruction-tuning
- sft
- chatml
- kapinstruct
- text-generation
datasets:
- kaptaan45/KapCode-1B
- kaptaan45/KapInstruct-100M
pipeline_tag: text-generation
library_name: transformers
---

# QaptaanLM-0.75B-Instruct: Efficient Hybrid-Attention Code & Reasoning Assistant

[![License](https://img.shields.io/badge/License-Apache%202.0-green.svg)](https://opensource.org/licenses/Apache-2.0)
[![Parameters](https://img.shields.io/badge/Parameters-752M%20(Text--Only)-blue.svg)](#model-specification)
[![Architecture](https://img.shields.io/badge/Architecture-Hybrid%20DeltaNet%20%2B%20GQA-purple.svg)](#architecture)
[![Context Length](https://img.shields.io/badge/Context-256K%20Native-orange.svg)](#model-specification)
[![GitHub](https://img.shields.io/badge/GitHub-QaptaanLM--0.75B-181717.svg?logo=github)](https://github.com/rudy-07/QaptaanLM-0.75B)
[![Kaggle Model](https://img.shields.io/badge/Kaggle-Model-20BEFF.svg?logo=kaggle)](https://www.kaggle.com/models/kaptaan45/qaptaanlm-0.75b)
[![SFT Dataset](https://img.shields.io/badge/%F0%9F%A4%97%20SFT%20Dataset-kaptaan45%2FKapInstruct--100M-orange.svg)](https://huggingface.co/datasets/kaptaan45/KapInstruct-100M)

**QaptaanLM-0.75B-Instruct** is the official instruction-tuned model of the **QaptaanLM-0.75B** family, optimized for Python code generation, bug fixing, SQL query formulation, and multi-turn technical dialogue.

It is trained through a rigorous two-stage curriculum:
1. **Stage 1: Continued Pre-Training (CPT)** on **[KapCode-1B](https://huggingface.co/datasets/kaptaan45/KapCode-1B)** (1B high-signal code, doc, and STEM tokens with 50% FIM infilling).
2. **Stage 2: Supervised Fine-Tuning (SFT)** on **[KapInstruct-100M](https://huggingface.co/datasets/kaptaan45/KapInstruct-100M)** (100M tokens across 12 balanced instruction datasets) formatted with **Qwen ChatML** and **strict assistant-only loss masking**.

---

## Model Specification

| Property | Value | Notes |
| :--- | :--- | :--- |
| **Model Name** | QaptaanLM-0.75B-Instruct | Text-only instruction-aligned model |
| **Base Architecture** | `Qwen/Qwen3.5-0.8B-Base` | Stripped vision transformer, 100% text capacity |
| **Total Parameters** | **752,382,976 (752M)** | Text-only dense parameters |
| **Hidden Size ($d_{model}$)** | 1024 | Base hidden dimension |
| **Intermediate Size ($d_{ffn}$)** | 3584 | SwiGLU non-linear activation |
| **Total Layers** | 24 | 18 Linear Attention + 6 Full GQA layers (3:1 ratio) |
| **Max Context Window** | 262,144 tokens (256K native) | Powered by interleaved M-RoPE ($\theta = 10,000,000$) |
| **Prompt Format** | Qwen ChatML (`<|im_start|>` / `<|im_end|>`) | Standard system / user / assistant dialogue |
| **Precision** | `bfloat16`, `float16`, `float32` | Hardware accelerated Tensor Cores / TPUs |

---

## Quickstart & Usage

### 1. ChatML Python Code Generation

```python
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "kaptaan45/QaptaanLM-0.75B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

messages = [
    {"role": "system", "content": "You are QaptaanLM, an expert programming and reasoning assistant."},
    {"role": "user", "content": "Write a Python function `is_palindrome(s: str) -> bool` that returns True if s is a palindrome."}
]

chat_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(response)
```

### 2. Direct ChatML String Format

```python
prompt = "Fix the bug in this Python function:\n```python\ndef append_item(val, items=[]):\n    items.append(val)\n    return items\n```"
chat_text = f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"

inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        eos_token_id=[tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids(\"<|im_end|>\")],
    )
print(tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))
```

---

## SFT Training Mixture (KapInstruct-100M)

KapInstruct-100M unifies 12 instruction domains aligned with assistant-only cross-entropy loss masking:
- **Code Generation [31%]**: Magicoder-Evol (13%), Magicoder-OSS (8%), Self-OSS-Instruct (5%), Smol-Constraints (3%)
- **General Dialogue & Reasoning [27%]**: Smol-Magpie-Ultra (18%), OpenHermes-2.5 (9%)
- **Mathematical Reasoning (CoT) [17%]**: OpenMathInstruct-2 (11%), NuminaMath-CoT (6%)
- **Debugging & Error Repair [10%]**: CodeFeedback-Filtered (10%)
- **STEM & Scientific QA [11%]**: OpenThoughts-114k (7%), WebInstructSub (4%)
- **Strict Constraint Adherence [4%]**: Tulu-3-SFT (6%), Smol-Constraints (3%)

---

## Citation

```bibtex
@misc{qaptaanlm_instruct2026,
  title   = {{QaptaanLM-0.75B-Instruct}: High-Efficiency Hybrid Attention Instruction Model},
  author  = {Rudy and Contributors},
  year    = {2026},
  publisher = {Hugging Face},
  url     = {https://huggingface.co/kaptaan45/QaptaanLM-0.75B-Instruct}
}
```

## License

Released under the [Apache 2.0 License](https://opensource.org/licenses/Apache-2.0).
'''

with open(export_dir / "README.md", "w", encoding="utf-8") as f:
    f.write(instruct_model_card)
print("✓ Created comprehensive README.md (Model Card)")


## 5. Local Validation & Smoke Test Inference

In [ ]:
print("=" * 75)
print("VALIDATING MODEL PACKAGE VIA PYTORCH AUTOCLASS")
print("=" * 75)

dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else (torch.float16 if torch.cuda.is_available() else torch.float32)
val_tokenizer = AutoTokenizer.from_pretrained(str(export_dir), trust_remote_code=True)
val_model = AutoModelForCausalLM.from_pretrained(
    str(export_dir),
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)
val_model.eval()

stop_ids = [val_tokenizer.eos_token_id]
im_end = val_tokenizer.convert_tokens_to_ids("<|im_end|>")
if im_end is not None and im_end not in stop_ids:
    stop_ids.append(im_end)

def test_prompt(p_text):
    chat = f"<|im_start|>user\n{p_text}<|im_end|>\n<|im_start|>assistant\n"
    inputs = val_tokenizer(chat, return_tensors="pt").to(val_model.device)
    with torch.no_grad():
        out = val_model.generate(
            **inputs,
            max_new_tokens=160,
            do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=stop_ids,
            pad_token_id=val_tokenizer.eos_token_id,
        )
    res = val_tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\nPROMPT : {p_text}")
    print(f"RESPONSE:\n{res.strip()}")
    print("-" * 70)

test_prompt("Write a Python function `is_palindrome(s: str) -> bool` that returns True if s is a palindrome.")
test_prompt("Write an SQL query to select all employees with salary greater than 50000 from the Employee table.")

print("\n✓ Local model validation successful!")
del val_model
del val_tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 6. Publish to Hugging Face Hub

In [ ]:
from huggingface_hub import HfApi, create_repo

TARGET_REPO = "kaptaan45/QaptaanLM-0.75B-Instruct"
print("=" * 75)
print(f"PUBLISHING TO HUGGING FACE HUB: {TARGET_REPO}")
print("=" * 75)

api = HfApi(token=HF_TOKEN)

# Create repo if it doesn't exist
try:
    create_repo(TARGET_REPO, repo_type="model", private=False, token=HF_TOKEN, exist_ok=True)
    print(f"✓ Repository {TARGET_REPO} ready on Hugging Face Hub.")
except Exception as e:
    print(f"[INFO] Repo status: {e}")

# Upload all repository files
print(f"Uploading folder {export_dir} to {TARGET_REPO}...")
upload_info = api.upload_folder(
    folder_path=str(export_dir),
    repo_id=TARGET_REPO,
    repo_type="model",
    commit_message="feat: upload official QaptaanLM-0.75B-Instruct model with custom hybrid modeling and Model Card",
)

print("\n" + "=" * 75)
print(f"🎉 MODEL SUCCESSFULLY PUBLISHED TO HUGGING FACE!")
print(f"🔗 Hub URL: https://huggingface.co/{TARGET_REPO}")
print("=" * 75)
